# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
import os

# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"

# download only if the file is not already there
if not os.path.exists(jan_2019_trip_data):
    response = requests.get(download_url)
    if response.status_code == 200:
        with open(jan_2019_trip_data, "wb") as f:
            f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [5]:
from pyspark.sql import functions as F

#### 1.1 Unique key for each trip

In [6]:
# unique id for each trip
df_trips = df_trips.withColumn("trip_id", F.monotonically_increasing_id())

# trip duration in minutes (used in several questions)
df_trips = df_trips.withColumn(
    "duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime")
     - F.unix_timestamp("tpep_pickup_datetime")) / 60
)

df_trips.select("trip_id", "tpep_pickup_datetime",
                "tpep_dropoff_datetime", "duration_min").show(5)

+-----------+--------------------+---------------------+------------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|      duration_min|
+-----------+--------------------+---------------------+------------------+
|34359738368| 2019-01-01 00:46:40|  2019-01-01 00:53:20| 6.666666666666667|
|34359738369| 2019-01-01 00:59:47|  2019-01-01 01:18:59|              19.2|
|34359738370| 2018-12-21 13:48:30|  2018-12-21 13:52:40| 4.166666666666667|
|34359738371| 2018-11-28 15:52:25|  2018-11-28 15:55:45|3.3333333333333335|
|34359738372| 2018-11-28 15:56:57|  2018-11-28 15:58:33|               1.6|
+-----------+--------------------+---------------------+------------------+
only showing top 5 rows


The ids are unique but not consecutive: each partition gets its own range.

#### 1.2 Highest and average passenger count

In [7]:
df_trips.orderBy(F.desc("passenger_count")) \
    .select("trip_id", "passenger_count", "trip_distance", "total_amount") \
    .show(5)

df_trips.agg(F.avg("passenger_count").alias("avg_passengers")).show()

+-----------+---------------+-------------+------------+
|    trip_id|passenger_count|trip_distance|total_amount|
+-----------+---------------+-------------+------------+
|34361750466|            9.0|          0.0|        11.3|
|34367025051|            9.0|          0.0|        10.3|
|34362622363|            9.0|          0.0|       12.25|
|34361034655|            9.0|          0.0|         9.3|
|34364273075|            9.0|          0.0|      110.76|
+-----------+---------------+-------------+------------+
only showing top 5 rows


+------------------+
|    avg_passengers|
+------------------+
|1.5670317144945614|
+------------------+



Max is 9 passengers (several trips, all with a distance of 0, so probably errors). The average is about 1.57, most people ride alone.

#### 1.3 Shortest / longest trip by distance and by time

In [8]:
# by distance
df_trips.orderBy("trip_distance") \
    .select("trip_id", "trip_distance", "duration_min").show(3)
df_trips.orderBy(F.desc("trip_distance")) \
    .select("trip_id", "trip_distance", "duration_min").show(3)

# shortest trip with a distance above 0
df_trips.filter(F.col("trip_distance") > 0).orderBy("trip_distance") \
    .select("trip_id", "trip_distance", "duration_min").show(3)

+-----------+-------------+------------------+
|    trip_id|trip_distance|      duration_min|
+-----------+-------------+------------------+
|34359738372|          0.0|               1.6|
|34359738373|          0.0|2.6166666666666667|
|34359738374|          0.0|               4.1|
+-----------+-------------+------------------+
only showing top 3 rows


+-----------+-------------+-----------------+
|    trip_id|trip_distance|     duration_min|
+-----------+-------------+-----------------+
|34365812459|        831.8|9.483333333333333|
|34364025001|        700.7|6.933333333333334|
|34366509353|       214.01|673.0833333333334|
+-----------+-------------+-----------------+
only showing top 3 rows


+-----------+-------------+-------------------+
|    trip_id|trip_distance|       duration_min|
+-----------+-------------+-------------------+
|34359750932|         0.01|0.06666666666666667|
|34359757196|         0.01| 0.6833333333333333|
|34359752687|         0.01| 0.4666666666666667|
+-----------+-------------+-------------------+
only showing top 3 rows


In [9]:
# by time
df_trips.orderBy("duration_min") \
    .select("trip_id", "tpep_pickup_datetime",
            "tpep_dropoff_datetime", "duration_min").show(3)
df_trips.orderBy(F.desc("duration_min")) \
    .select("trip_id", "tpep_pickup_datetime",
            "tpep_dropoff_datetime", "duration_min").show(3)

+-----------+--------------------+---------------------+-------------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|       duration_min|
+-----------+--------------------+---------------------+-------------------+
|34360941552| 2019-01-06 15:15:08|  2018-11-09 02:34:38|           -84280.5|
|34362714661| 2019-01-13 15:15:27|  2018-12-25 07:37:43|-27817.733333333334|
|34364480114| 2019-01-20 15:15:30|  2019-01-18 15:57:16| -2838.233333333333|
+-----------+--------------------+---------------------+-------------------+
only showing top 3 rows


+-----------+--------------------+---------------------+------------------+
|    trip_id|tpep_pickup_datetime|tpep_dropoff_datetime|      duration_min|
+-----------+--------------------+---------------------+------------------+
|34359806635| 2019-01-01 07:01:20|  2019-01-31 14:29:21| 43648.01666666667|
|34360330630| 2019-01-03 22:24:36|  2019-01-27 10:41:17|33856.683333333334|
|34360614224| 2019-01-05 04:21:40|  2019-01-27 01:53:46|           31532.1|
+-----------+--------------------+---------------------+------------------+
only showing top 3 rows


- Distance: many trips at 0 miles, shortest real trip is 0.01 mile. Longest is 831.8 miles in 9 minutes: impossible.
- Time: shortest has a negative duration, longest lasts about 30 days.

These extremes are data errors (see 1.9).

#### 1.4 Busiest / slowest day

In [10]:
# keep only January 2019 (some dates are in 2008, 2018, 2088...)
df_jan = df_trips.filter(
    (F.col("tpep_pickup_datetime") >= "2019-01-01")
    & (F.col("tpep_pickup_datetime") < "2019-02-01")
)

df_days = df_jan.groupBy(
    F.to_date("tpep_pickup_datetime").alias("day")
).count()

df_days.orderBy(F.desc("count")).show(3)
df_days.orderBy("count").show(3)

+----------+------+
|       day| count|
+----------+------+
|2019-01-25|292499|
|2019-01-11|291714|
|2019-01-31|284625|
+----------+------+
only showing top 3 rows


+----------+------+
|       day| count|
+----------+------+
|2019-01-01|189432|
|2019-01-21|192826|
|2019-01-02|198737|
+----------+------+
only showing top 3 rows


Busiest: Friday January 25 (292,499 trips). Slowest: January 1 (189,432), a holiday. January 21 (Martin Luther King Day) is second slowest.

#### 1.5 Busiest / slowest time of day

In [11]:
df_hours = df_jan.groupBy(
    F.hour("tpep_pickup_datetime").alias("hour")
).count()

df_hours.orderBy(F.desc("count")).show(3)
df_hours.orderBy("count").show(3)

+----+------+
|hour| count|
+----+------+
|  18|515374|
|  19|475152|
|  17|468407|
+----+------+
only showing top 3 rows


+----+-----+
|hour|count|
+----+-----+
|   4|61423|
|   5|75532|
|   3|78084|
+----+-----+
only showing top 3 rows


In [12]:
df_periods = df_jan.withColumn("hour", F.hour("tpep_pickup_datetime")) \
    .withColumn(
        "period",
        F.when((F.col("hour") >= 6) & (F.col("hour") < 12), "morning")
         .when((F.col("hour") >= 12) & (F.col("hour") < 18), "afternoon")
         .when(F.col("hour") >= 18, "evening")
         .otherwise("late night")
    )

df_periods.groupBy("period").count().orderBy(F.desc("count")).show()

+----------+-------+
|    period|  count|
+----------+-------+
| afternoon|2580328|
|   evening|2474365|
|   morning|1959935|
|late night| 681452|
+----------+-------+



Busiest hour: 18h (end of work day). Slowest: 4h. By period, afternoon is the busiest and late night (0h to 6h) the slowest.

#### 1.6 Busiest / slowest day of the week on average

In [13]:
# January has 4 or 5 of each weekday, so we average per day
df_days.withColumn("weekday", F.date_format("day", "EEEE")) \
    .groupBy("weekday") \
    .agg(F.avg("count").alias("avg_trips")) \
    .orderBy(F.desc("avg_trips")) \
    .show()

+---------+---------+
|  weekday|avg_trips|
+---------+---------+
|   Friday| 271787.5|
| Thursday| 271398.4|
|Wednesday| 253045.8|
| Saturday|252494.75|
|  Tuesday| 241815.2|
|   Monday| 226941.0|
|   Sunday| 214972.5|
+---------+---------+



Average and not total, because January 2019 has 5 Tuesdays/Wednesdays/Thursdays but 4 of the other days. Friday is the busiest, Sunday the slowest.

#### 1.7 Does distance or number of passengers affect the tip?

In [14]:
# average tip by payment type (1 = credit card, 2 = cash)
df_trips.groupBy("payment_type") \
    .agg(F.avg("tip_amount").alias("avg_tip")) \
    .orderBy("payment_type").show()

+------------+--------------------+
|payment_type|             avg_tip|
+------------+--------------------+
|           0|0.061789899553571406|
|           1|  2.5539784713276794|
|           2|3.146533815927865...|
|           3|-0.00157138121213...|
|           4|0.003771607702642...|
+------------+--------------------+



In [15]:
# cash tips are not recorded, so we keep only card payments
df_card = df_trips.filter(F.col("payment_type") == 1)

df_card.select(
    F.corr("trip_distance", "tip_amount").alias("corr_distance_tip"),
    F.corr("passenger_count", "tip_amount").alias("corr_passengers_tip")
).show()

df_card.groupBy("passenger_count") \
    .agg(F.avg("tip_amount").alias("avg_tip"),
         F.count("*").alias("nb_trips")) \
    .orderBy("passenger_count").show()

+------------------+--------------------+
| corr_distance_tip| corr_passengers_tip|
+------------------+--------------------+
|0.6715943173994262|0.010141706333933796|
+------------------+--------------------+



+---------------+------------------+--------+
|passenger_count|           avg_tip|nb_trips|
+---------------+------------------+--------+
|            0.0| 2.519602931459125|   83235|
|            1.0|2.5339521622183447| 3936888|
|            2.0| 2.614486572689903|  781318|
|            3.0| 2.585738670425347|  218521|
|            4.0|2.6029785593256283|   92068|
|            5.0|2.6182445012300475|  231279|
|            6.0| 2.609179752008406|  142908|
|            7.0| 11.30090909090909|      11|
|            8.0|  6.96074074074074|      27|
|            9.0|           3.50625|       8|
+---------------+------------------+--------+



Cash tips are almost 0 in the data (not recorded), so we only use card payments.
- Distance: yes, correlation 0.67, longer trips get bigger tips.
- Passengers: no, correlation 0.01, the average tip stays around 2.5 to 2.6 dollars (7 to 9 passengers: too few trips to conclude).

#### 1.8 Highest "extra" charge

In [16]:
df_trips.orderBy(F.desc("extra")) \
    .select("trip_id", "extra", "fare_amount", "total_amount",
            "tpep_pickup_datetime").show(3)

+-----------+------+-----------+------------+--------------------+
|    trip_id| extra|fare_amount|total_amount|tpep_pickup_datetime|
+-----------+------+-----------+------------+--------------------+
|34365061851|535.38|  355676.98|   356214.78| 2019-01-23 08:58:09|
|34367191598| 23.04|        4.5|       28.34| 2019-01-31 10:06:09|
|34360281571|  18.5|       61.0|        92.3| 2019-01-03 18:32:36|
+-----------+------+-----------+------------+--------------------+
only showing top 3 rows


Highest extra: 535.38 dollars, on a trip with a fare of 355,676.98 dollars, so an error. A normal extra is 0.5 or 1 dollar (rush hour / night).

#### 1.9 Strange data points / outliers

In [17]:
print("passenger_count = 0:",
      df_trips.filter(F.col("passenger_count") == 0).count())
print("passenger_count null:",
      df_trips.filter(F.col("passenger_count").isNull()).count())
print("trip_distance = 0:",
      df_trips.filter(F.col("trip_distance") == 0).count())
print("fare_amount < 0:",
      df_trips.filter(F.col("fare_amount") < 0).count())
print("duration <= 0:",
      df_trips.filter(F.col("duration_min") <= 0).count())
print("duration > 24h:",
      df_trips.filter(F.col("duration_min") > 24 * 60).count())
print("not in January 2019:", df_trips.count() - df_jan.count())

df_trips.select("trip_distance", "fare_amount", "tip_amount",
                "extra", "passenger_count", "duration_min").describe().show()

passenger_count = 0: 117381
passenger_count null: 28672


trip_distance = 0: 55089


fare_amount < 0: 7129


duration <= 0: 6557


duration > 24h: 5


not in January 2019: 537


+-------+------------------+-----------------+------------------+------------------+------------------+------------------+
|summary|     trip_distance|      fare_amount|        tip_amount|             extra|   passenger_count|      duration_min|
+-------+------------------+-----------------+------------------+------------------+------------------+------------------+
|  count|           7696617|          7696617|           7696617|           7696617|           7667945|           7696617|
|   mean|2.8301461681153532|12.52967677747685|1.8208300763883147|0.3374054146126797|1.5670317144945614|16.551081570422276|
| stddev| 3.774548394256295|261.5897471783846|2.4994631914320986|0.5313564053935059|1.2244198591042095| 81.67539611217202|
|    min|               0.0|           -362.0|             -63.5|             -60.0|               0.0|          -84280.5|
|    max|             831.8|        623259.86|            787.25|            535.38|               9.0| 43648.01666666667|
+-------+-------

- **0 passengers** (117,381) or empty (28,672): a trip needs at least 1 passenger, the driver did not enter it.
- **Distance 0** (55,089): cancelled trip or GPS problem. 831.8 miles in 9 minutes is impossible.
- **Negative fares** (7,129): refunds or corrections.
- **Duration 0 or negative** (6,557) and **over 24h** (5): meter not stopped correctly.
- **537 trips outside January 2019** (years 2001, 2008, 2088...): wrong clock.
- **Fare of 623,259.86 dollars** and extra of 535.38: typing errors.

These rows should be removed for a real analysis. For time questions we already use `df_jan`.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

#### 2.0 Load the taxi zone lookup

In [18]:
zones_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
zones_file = "taxi_zone_lookup.csv"

if not os.path.exists(zones_file):
    response = requests.get(zones_url)
    if response.status_code == 200:
        with open(zones_file, "wb") as f:
            f.write(response.content)

df_zones = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(zones_file)

df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


#### 2.1 Join trips with boroughs

In [19]:
# one join for the pick-up borough, one for the drop-off borough
zones_pu = df_zones.select(F.col("LocationID").alias("PULocationID"),
                           F.col("Borough").alias("pu_borough"))
zones_do = df_zones.select(F.col("LocationID").alias("DOLocationID"),
                           F.col("Borough").alias("do_borough"))

df_z = df_jan.join(zones_pu, "PULocationID", "left") \
             .join(zones_do, "DOLocationID", "left")

df_z.select("trip_id", "pu_borough", "do_borough").show(5)

+-----------+----------+----------+
|    trip_id|pu_borough|do_borough|
+-----------+----------+----------+
|34359738368| Manhattan| Manhattan|
|34359738369| Manhattan| Manhattan|
|34359738375| Manhattan| Manhattan|
|34359738376| Manhattan|    Queens|
|34359738377| Manhattan| Manhattan|
+-----------+----------+----------+
only showing top 5 rows


Left join to keep all trips. `Unknown` is zone 264 (location not known) and `N/A` is zone 265 (outside of NYC).

#### 2.2 Borough with most pickups / dropoffs

In [20]:
df_z.groupBy("pu_borough").count().orderBy(F.desc("count")).show()
df_z.groupBy("do_borough").count().orderBy(F.desc("count")).show()

+-------------+-------+
|   pu_borough|  count|
+-------------+-------+
|    Manhattan|6950511|
|       Queens| 471113|
|      Unknown| 159807|
|     Brooklyn|  91896|
|        Bronx|  18056|
|          N/A|   3890|
|          EWR|    446|
|Staten Island|    361|
+-------------+-------+



+-------------+-------+
|   do_borough|  count|
+-------------+-------+
|    Manhattan|6816936|
|       Queens| 340914|
|     Brooklyn| 301074|
|      Unknown| 149091|
|        Bronx|  58068|
|          N/A|  16900|
|          EWR|  10913|
|Staten Island|   2184|
+-------------+-------+



Manhattan by far: about 90% of pickups and dropoffs. Then Queens (airports JFK and LaGuardia). Staten Island is last.

#### 2.3 Busy / slow times by borough

In [21]:
df_z = df_z.withColumn("hour", F.hour("tpep_pickup_datetime")) \
    .withColumn(
        "period",
        F.when((F.col("hour") >= 6) & (F.col("hour") < 12), "morning")
         .when((F.col("hour") >= 12) & (F.col("hour") < 18), "afternoon")
         .when(F.col("hour") >= 18, "evening")
         .otherwise("late night")
    )

df_z.groupBy("pu_borough") \
    .pivot("period", ["morning", "afternoon", "evening", "late night"]) \
    .count() \
    .orderBy(F.desc("afternoon")) \
    .show()

+-------------+-------+---------+-------+----------+
|   pu_borough|morning|afternoon|evening|late night|
+-------------+-------+---------+-------+----------+
|    Manhattan|1774040|  2338654|2232164|    605653|
|       Queens| 107769|   157374| 163067|     42903|
|      Unknown|  39866|    54858|  50990|     14093|
|     Brooklyn|  29426|    22673|  23998|     15799|
|        Bronx|   7671|     5352|   2866|      2167|
|          N/A|    905|     1057|   1168|       760|
|          EWR|    108|      248|     60|        30|
|Staten Island|    150|      112|     52|        47|
+-------------+-------+---------+-------+----------+



Manhattan and Queens are busiest in the afternoon/evening. Brooklyn and the Bronx are busiest in the morning (trips to work). Late night is the slowest everywhere.

#### 2.4 Busiest days of the week by borough

In [22]:
days = ["Monday", "Tuesday", "Wednesday", "Thursday",
        "Friday", "Saturday", "Sunday"]

# average number of trips per day, like in 1.6
df_z.groupBy("pu_borough",
             F.to_date("tpep_pickup_datetime").alias("day")).count() \
    .withColumn("weekday", F.date_format("day", "EEEE")) \
    .groupBy("pu_borough") \
    .pivot("weekday", days) \
    .agg(F.round(F.avg("count"))) \
    .orderBy(F.desc("Friday")) \
    .show()

+-------------+--------+--------+---------+--------+--------+--------+--------+
|   pu_borough|  Monday| Tuesday|Wednesday|Thursday|  Friday|Saturday|  Sunday|
+-------------+--------+--------+---------+--------+--------+--------+--------+
|    Manhattan|201858.0|217239.0| 228953.0|245903.0|246224.0|231875.0|192553.0|
|       Queens| 16662.0| 15737.0|  15163.0| 15792.0| 16038.0| 11915.0| 14798.0|
|      Unknown|  5354.0|  4905.0|   5156.0|  5785.0|  5441.0|  5176.0|  4174.0|
|     Brooklyn|  2377.0|  3156.0|   3020.0|  3143.0|  3273.0|  2901.0|  2775.0|
|        Bronx|   543.0|   612.0|    600.0|   624.0|   667.0|   482.0|   528.0|
|          N/A|   129.0|   141.0|    127.0|   127.0|   111.0|   123.0|   117.0|
|          EWR|     8.0|    15.0|     17.0|    12.0|    19.0|    14.0|    17.0|
|Staten Island|    11.0|    11.0|     10.0|    12.0|    16.0|    10.0|    11.0|
+-------------+--------+--------+---------+--------+--------+--------+--------+



Manhattan: Thursday/Friday busiest, Sunday slowest. Queens: Monday busiest, Saturday slowest. Brooklyn and Bronx: Friday busiest.

#### 2.5 Average trip distance and fare by borough

In [23]:
df_z.groupBy("pu_borough") \
    .agg(F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
         F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
         F.count("*").alias("nb_trips")) \
    .orderBy(F.desc("avg_fare")) \
    .show()

+-------------+------------+--------+--------+
|   pu_borough|avg_distance|avg_fare|nb_trips|
+-------------+------------+--------+--------+
|          EWR|        2.64|   76.24|     446|
|          N/A|        3.19|   59.57|    3890|
|Staten Island|        12.5|   45.29|     361|
|       Queens|       11.28|   35.14|  471113|
|        Bronx|        7.23|   26.27|   18056|
|     Brooklyn|        4.79|   18.65|   91896|
|      Unknown|        2.42|   14.94|  159807|
|    Manhattan|        2.23|   10.79| 6950511|
+-------------+------------+--------+--------+



Manhattan has the shortest and cheapest trips (2.23 miles, 10.79 dollars). Queens (airports) and Staten Island (far from Manhattan) have the longest trips, over 11 miles. EWR (Newark airport) has the highest fare, but only 446 trips.

#### 2.6 Highest / lowest fare and their borough

In [24]:
cols = ["trip_id", "fare_amount", "trip_distance", "pu_borough", "do_borough"]

df_z.orderBy(F.desc("fare_amount")).select(cols).show(3)
df_z.orderBy("fare_amount").select(cols).show(3)

+-----------+-----------+-------------+----------+----------+
|    trip_id|fare_amount|trip_distance|pu_borough|do_borough|
+-----------+-----------+-------------+----------+----------+
|34362238023|  623259.86|          2.4| Manhattan| Manhattan|
|34365061851|  355676.98|          0.0| Manhattan|   Unknown|
|34361898339|    36090.3|          0.0|   Unknown|   Unknown|
+-----------+-----------+-------------+----------+----------+
only showing top 3 rows


+-----------+-----------+-------------+----------+----------+
|    trip_id|fare_amount|trip_distance|pu_borough|do_borough|
+-----------+-----------+-------------+----------+----------+
|34364629017|     -362.0|          0.0|    Queens|    Queens|
|34366046568|     -320.0|          0.0|       N/A|       N/A|
|34359795462|     -300.0|          0.0|     Bronx|     Bronx|
+-----------+-----------+-------------+----------+----------+
only showing top 3 rows


Highest: 623,259.86 dollars for 2.4 miles in Manhattan, an error. Lowest: -362 dollars in Queens, a refund. See outliers in 1.9.

#### 2.7 Compare with the most recent January (2026)

The most recent January available on the TLC website is January 2026.

In [25]:
jan_2026_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet'
jan_2026_trip_data = "yellow_tripdata_2026-01.parquet"

if not os.path.exists(jan_2026_trip_data):
    response = requests.get(jan_2026_url)
    if response.status_code == 200:
        with open(jan_2026_trip_data, "wb") as f:
            f.write(response.content)

df_2026 = spark.read.parquet(jan_2026_trip_data)
df_2026 = df_2026.withColumn(
    "duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime")
     - F.unix_timestamp("tpep_pickup_datetime")) / 60
)

In [26]:
# same simple cleaning for both years (the outliers change the averages)
def average_metrics(df, year):
    df_clean = df.filter(
        (F.col("fare_amount") > 0) & (F.col("fare_amount") < 500)
        & (F.col("trip_distance") > 0) & (F.col("trip_distance") < 100)
        & (F.col("duration_min") > 0) & (F.col("duration_min") < 180)
    )
    return df_clean.agg(
        F.lit(year).alias("year"),
        F.count("*").alias("nb_trips"),
        F.round(F.avg("passenger_count"), 2).alias("avg_passengers"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.avg("duration_min"), 2).alias("avg_duration_min"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
        F.round(F.avg("total_amount"), 2).alias("avg_total")
    )

average_metrics(df_trips, 2019).union(average_metrics(df_2026, 2026)).show()

+----+--------+--------------+------------+----------------+--------+-------+---------+
|year|nb_trips|avg_passengers|avg_distance|avg_duration_min|avg_fare|avg_tip|avg_total|
+----+--------+--------------+------------+----------------+--------+-------+---------+
|2019| 7613584|          1.57|        2.85|           12.99|   12.29|   1.81|    15.56|
|2026| 3515230|          1.25|        3.49|           17.11|   21.07|   2.67|    29.66|
+----+--------+--------------+------------+----------------+--------+-------+---------+



Yes, all averages changed between 2019 and 2026:
- **Half as many trips** (7.6M to 3.5M), probably because of Uber/Lyft.
- **Fewer passengers** (1.57 to 1.25), **longer trips** (2.85 to 3.49 miles, 13 to 17 minutes).
- **Much more expensive**: fare 12.29 to 21.07 dollars, total 15.56 to 29.66 dollars (new fees, like the `cbd_congestion_fee` column added in 2025).

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

SQL on the same data: we register the DataFrames as tables.

In [27]:
df_jan.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

#### 3.1 Highest and average passenger count (same as 1.2)

In [28]:
spark.sql("""
    SELECT MAX(passenger_count) AS max_passengers,
           ROUND(AVG(passenger_count), 2) AS avg_passengers
    FROM trips
""").show()

+--------------+--------------+
|max_passengers|avg_passengers|
+--------------+--------------+
|           9.0|          1.57|
+--------------+--------------+



#### 3.2 Busiest time of day (same as 1.5)

In [29]:
spark.sql("""
    SELECT HOUR(tpep_pickup_datetime) AS hour,
           COUNT(*) AS nb_trips
    FROM trips
    GROUP BY HOUR(tpep_pickup_datetime)
    ORDER BY nb_trips DESC
    LIMIT 3
""").show()

+----+--------+
|hour|nb_trips|
+----+--------+
|  18|  515374|
|  19|  475152|
|  17|  468407|
+----+--------+



#### 3.3 Borough with most pickups, with a join (same as 2.2)

In [30]:
spark.sql("""
    SELECT z.Borough AS pu_borough,
           COUNT(*) AS nb_pickups
    FROM trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY nb_pickups DESC
""").show()

+-------------+----------+
|   pu_borough|nb_pickups|
+-------------+----------+
|    Manhattan|   6950511|
|       Queens|    471113|
|      Unknown|    159807|
|     Brooklyn|     91896|
|        Bronx|     18056|
|          N/A|      3890|
|          EWR|       446|
|Staten Island|       361|
+-------------+----------+



Same results as in Python: both go through the same Catalyst optimizer, only the syntax changes.

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing